In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd  /content/drive/MyDrive/MedGen

/content/drive/MyDrive/MedGen


In [ ]:
!pip install netron

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 14.8 MB/s eta 0:00:00


In [ ]:
# Cell 1: Import necessary libraries
import torch
import numpy as np
from modules.tokenizers import Tokenizer
from modules.dataloaders import R2DataLoader
from modules.loss import compute_loss
from modules.optimizers import build_optimizer, build_lr_scheduler
from modules.trainer import Trainer
from modules.tester import Tester
from models.r2gen import R2GenModel
import argparse
import os

# Set deterministic behavior for reproducibility
torch.manual_seed(9233)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(9233)


In [ ]:
# Cell 2: Define and parse arguments manually
class Args:
    # Data input settings
    image_dir = 'data/iu_xray/images/'
    ann_path = 'data/iu_xray/annotation.json'

    # Data loader settings
    dataset_name = 'iu_xray'
    max_seq_length = 60
    threshold = 3
    num_workers = 2
    batch_size = 4

    # Model settings
    visual_extractor = 'resnet101'
    visual_extractor_pretrained = True
    d_model = 512
    d_ff = 512
    d_vf = 2048
    num_heads = 8
    num_layers = 3
    dropout = 0.1
    logit_layers = 1
    bos_idx = 0
    eos_idx = 0
    pad_idx = 0
    use_bn = 0
    drop_prob_lm = 0.5
    rm_num_slots = 3
    rm_num_heads = 8
    rm_d_model = 512

    # Sample related
    sample_method = 'beam_search'
    beam_size = 3
    temperature = 1.0
    sample_n = 1
    group_size = 1
    output_logsoftmax = 1
    decoding_constraint = 0
    block_trigrams = 1

    # Trainer settings
    n_gpu = 1
    epochs = 10
    save_dir = 'results/iu_xray'
    record_dir = 'records/'
    save_period = 1
    monitor_mode = 'max'
    early_stop = 50

    # Optimization
    optim = 'Adam'
    lr_ve = 5e-5
    lr_ed = 1e-4
    weight_decay = 5e-5
    amsgrad = True

    # Learning Rate Scheduler
    lr_scheduler = 'StepLR'
    step_size = 50
    gamma = 0.1
    monitor_mode = 'max'
    monitor_metric = 'loss'

    # Others
    seed = 9233
    resume = None
    load = 'results/iu_xray/current_checkpoint.pth'

args = Args()


In [ ]:
# Cell 3: Initialize tokenizer
from modules.tokenizers import Tokenizer

tokenizer = Tokenizer(args)
print("Tokenizer initialized successfully!")


Tokenizer initialized successfully!


In [ ]:
# Cell 4: Create data loaders
from modules.dataloaders import R2DataLoader

train_dataloader = R2DataLoader(args, tokenizer, split='train', shuffle=True)
val_dataloader = R2DataLoader(args, tokenizer, split='val', shuffle=False)
test_dataloader = R2DataLoader(args, tokenizer, split='test', shuffle=False)

print("Data loaders created successfully!")
print(f"Training samples: {len(train_dataloader)}")
print(f"Validation samples: {len(val_dataloader)}")
print(f"Test samples: {len(test_dataloader)}")


Data loaders created successfully!
Training samples: 518
Validation samples: 74
Test samples: 148


In [ ]:
!pip install torchvision ipywidgets matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image
import ipywidgets as widgets
from IPython.display import display

In [ ]:
class GradCAM:
    def _init_(self, model, target_layer):
        self.model = model.eval()
        self.gradients = None
        self.activations = None

        def save_gradients_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        def save_activations_hook(module, input, output):
            self.activations = output.detach()

        target_layer.register_forward_hook(save_activations_hook)
        target_layer.register_backward_hook(save_gradients_hook)

    def generate(self, input_tensor, class_idx=None):
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax().item()
        score = output[0, class_idx]
        self.model.zero_grad()
        score.backward()

        gradients = self.gradients[0]
        activations = self.activations[0]
        weights = gradients.mean(dim=(1, 2), keepdim=True)
        cam = (weights * activations).sum(dim=0)
        cam = torch.clamp(cam, min=0).cpu().numpy()
        cam = cam - cam.min()
        cam = cam / cam.max()
        return cam

In [ ]:
resnet = models.resnet101(pretrained=True)
target_layer = resnet.layer4[-1]
gradcam = GradCAM()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet.to(device)

# Replace these with actual paths
generated_reports = pd.read_csv('/content/res (1) (1).csv', header=None, names=['generated'])
ground_truth_reports = pd.read_csv('/content/gts.csv', header=None, names=['ground_truth'])

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [ ]:
def show_sample(index):
    # Grab a batch from the test loader
    batch = next(iter(test_dataloader))

    # Assuming batch is a tuple: (images, labels) or (images, reports)
    images = batch[0]  # images tensor: [B, C, H, W]

    image = images[index].to(device).unsqueeze(0)  # add batch dimension

    # Generate Grad-CAM
    cam = gradcam.generate(image)

    image_np = image.squeeze(0).cpu().permute(1, 2, 0).numpy()
    image_np = np.clip(image_np, 0, 1)

    cam_resized = np.uint8(255 * cam)
    cam_resized = Image.fromarray(cam_resized).resize((224, 224))
    cam_resized = np.array(cam_resized)

    heatmap = plt.cm.jet(cam_resized / 255.0)[:, :, :3]
    overlay = heatmap * 0.5 + image_np
    overlay = np.clip(overlay, 0, 1)

    plt.figure(figsize=(5, 5))
    plt.imshow(overlay)
    plt.axis('off')
    plt.title(f"Sample {index}")
    plt.show()

    print("🔷 Generated Report:")
    print(generated_reports.loc[index, 'generated'])
    print("\n🔶 Ground Truth Report:")
    print(ground_truth_reports.loc[index, 'ground_truth'])

In [ ]:
batch = next(iter(test_dataloader))

for i, item in enumerate(batch):
    print(f"batch[{i}] -> type: {type(item)}")
    if isinstance(item, (list, tuple)):
        print(f"   length: {len(item)}")
        print(f"   type of first element: {type(item[0])}")
        if hasattr(item[0], 'shape'):
            print(f"   shape of first element: {item[0].shape}")

batch[0] -> type: <class 'tuple'>
   length: 4
   type of first element: <class 'str'>
batch[1] -> type: <class 'torch.Tensor'>
batch[2] -> type: <class 'torch.Tensor'>
batch[3] -> type: <class 'torch.Tensor'>


In [ ]:
def show_sample(index):
    batch = next(iter(test_dataloader))
    images = batch[1]  # This is your batch of image tensors
    image = images[index].to(device).unsqueeze(0)  # [1, C, H, W]

    # Generate Grad-CAM
    cam = gradcam.generate(image)

    # Convert tensor image to NumPy for display
    image_np = image.squeeze(0).cpu().permute(1, 2, 0).numpy()
    image_np = np.clip(image_np, 0, 1)

    cam_resized = np.uint8(255 * cam)
    cam_resized = Image.fromarray(cam_resized).resize((224, 224))
    cam_resized = np.array(cam_resized)

    heatmap = plt.cm.jet(cam_resized / 255.0)[:, :, :3]
    overlay = heatmap * 0.5 + image_np
    overlay = np.clip(overlay, 0, 1)

    plt.figure(figsize=(5, 5))
    plt.imshow(overlay)
    plt.axis('off')
    plt.title(f"Sample {index}")
    plt.show()

    print("🔷 Generated Report:")
    print(generated_reports.loc[index, 'generated'])
    print("\n🔶 Ground Truth Report:")
    print(ground_truth_reports.loc[index, 'ground_truth'])

In [ ]:
show_sample(0)

AttributeError: 'GradCAM' object has no attribute 'model'

In [ ]:
# Cell 5: Build model
from models.r2gen import R2GenModel

model = R2GenModel(args,tokenizer)
print("Model initialized successfully!")

model.load_state_dict(torch.load("/content/drive/MyDrive/MedGen/results/iu_xray/current_checkpoint.pth", map_location=device))
model = model.to(device)
model.eval()

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model initialized successfully!


RuntimeError: Error(s) in loading state_dict for R2GenModel:
	Missing key(s) in state_dict: "visual_extractor.model.0.weight", "visual_extractor.model.1.weight", "visual_extractor.model.1.bias", "visual_extractor.model.1.running_mean", "visual_extractor.model.1.running_var", "visual_extractor.model.4.0.conv1.weight", "visual_extractor.model.4.0.bn1.weight", "visual_extractor.model.4.0.bn1.bias", "visual_extractor.model.4.0.bn1.running_mean", "visual_extractor.model.4.0.bn1.running_var", "visual_extractor.model.4.0.conv2.weight", "visual_extractor.model.4.0.bn2.weight", "visual_extractor.model.4.0.bn2.bias", "visual_extractor.model.4.0.bn2.running_mean", "visual_extractor.model.4.0.bn2.running_var", "visual_extractor.model.4.0.conv3.weight", "visual_extractor.model.4.0.bn3.weight", "visual_extractor.model.4.0.bn3.bias", "visual_extractor.model.4.0.bn3.running_mean", "visual_extractor.model.4.0.bn3.running_var", "visual_extractor.model.4.0.downsample.0.weight", "visual_extractor.model.4.

In [ ]:
checkpoint = torch.load("/content/drive/MyDrive/MedGen/results/iu_xray/current_checkpoint.pth", map_location='cpu')

In [ ]:
print(type(checkpoint))
print(checkpoint.keys())

<class 'dict'>
dict_keys(['epoch', 'state_dict', 'optimizer', 'monitor_best'])


In [ ]:
import torch

# Load the full checkpoint
checkpoint = torch.load('/content/drive/MyDrive/MedGen/results/iu_xray/current_checkpoint.pth', map_location='cpu')

# Extract state_dict
state_dict = checkpoint['state_dict']

# Print a few keys to see the structure
print("All keys:")
for i, key in enumerate(state_dict.keys()):
    print(f"{i}: {key}")
    if i > 30:  # Only show first 30 to avoid huge output
        break

# Check for attention-related keys
attention_keys = [k for k in state_dict.keys() if 'attn' in k.lower() or 'attention' in k.lower()]
print("\nPossible attention-related keys:")
for key in attention_keys:
    print(key)

All keys:
0: visual_extractor.model.0.weight
1: visual_extractor.model.1.weight
2: visual_extractor.model.1.bias
3: visual_extractor.model.1.running_mean
4: visual_extractor.model.1.running_var
5: visual_extractor.model.1.num_batches_tracked
6: visual_extractor.model.4.0.conv1.weight
7: visual_extractor.model.4.0.bn1.weight
8: visual_extractor.model.4.0.bn1.bias
9: visual_extractor.model.4.0.bn1.running_mean
10: visual_extractor.model.4.0.bn1.running_var
11: visual_extractor.model.4.0.bn1.num_batches_tracked
12: visual_extractor.model.4.0.conv2.weight
13: visual_extractor.model.4.0.bn2.weight
14: visual_extractor.model.4.0.bn2.bias
15: visual_extractor.model.4.0.bn2.running_mean
16: visual_extractor.model.4.0.bn2.running_var
17: visual_extractor.model.4.0.bn2.num_batches_tracked
18: visual_extractor.model.4.0.conv3.weight
19: visual_extractor.model.4.0.bn3.weight
20: visual_extractor.model.4.0.bn3.bias
21: visual_extractor.model.4.0.bn3.running_mean
22: visual_extractor.model.4.0.bn3.r

In [ ]:
num_samples = len(generated_reports)
slider = widgets.IntSlider(min=0, max=num_samples-1, step=1, description='Sample:')

ui = widgets.VBox([slider])
out = widgets.interactive_output(show_sample, {'index': slider})

display(ui, out)

Output()

In [ ]:
import torch
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cv2
from torchvision import transforms
from torch.nn import Module
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:

from modules.visual_extractor import VisualExtractor  # your visual extractor
from models.r2gen import R2GenModel

model = R2GenModel(args,tokenizer)
print("Model initialized successfully!")
checkpoint = torch.load('/content/drive/MyDrive/MedGen/results/iu_xray/current_checkpoint.pth', map_location=device)
model.load_state_dict(checkpoint['state_dict'])
model.to(device)
model.eval()

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model initialized successfully!


R2GenModel(
  (visual_extractor): VisualExtractor(
    (model): Sequential(
      (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (4): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplac

In [ ]:
generated_df = pd.read_csv('/content/res (1) (1).csv')
groundtruth_df = pd.read_csv('/content/gts.csv')

# You already have test_loader prepared
# Ensure images are accessible by index
test_iter = iter(test_dataloader)

In [ ]:
class GradCAM:
    def _init_(self, model, target_layer):
        self.model = model
        self.model.eval()
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_backward_hook(backward_hook)

    def generate(self, input_tensor, class_idx=None):
        output = self.model.visual_extractor(input_tensor)  # only visual part
        logits = output.mean([2, 3])  # global average pooling

        if class_idx is None:
            class_idx = logits.argmax().item()

        self.model.zero_grad()
        one_hot = torch.zeros_like(logits)
        one_hot[0, class_idx] = 1
        logits.backward(gradient=one_hot)

        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        activations = self.activations[0]

        for i in range(pooled_gradients.shape[0]):
            activations[i, :, :] *= pooled_gradients[i]

        heatmap = torch.mean(activations, dim=0).cpu()
        heatmap = np.maximum(heatmap, 0)
        heatmap /= torch.max(heatmap)
        return heatmap.numpy()

In [ ]:
def apply_heatmap(image_tensor, heatmap):
    image = image_tensor.permute(1, 2, 0).cpu().numpy()
    image = np.uint8(255 * image)
    heatmap = cv2.resize(heatmap, (image.shape[1], image.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    cam = cv2.addWeighted(image, 0.6, heatmap, 0.4, 0)
    return cam

In [ ]:
def show_sample(index):
    batch = next(iter(test_dataloader))
    img_tensors = batch[0]  # tuple of images (T, C, H, W)
    imgs = img_tensors  # list or tuple of images
    image = imgs[0].unsqueeze(0).to(device)  # [1, C, H, W]

    gradcam = GradCAM(model, model.visual_extractor.model[-1])  # last conv layer
    heatmap = gradcam.generate(image)

    cam = apply_heatmap(image[0], heatmap)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 3, 1)
    plt.title('Original Image')
    plt.imshow(image[0].permute(1, 2, 0).cpu())
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.title('Grad-CAM')
    plt.imshow(cam)
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.title('Reports')
    plt.text(0.1, 0.6, f"Generated:\n{generated_df.iloc[index, 0]}", wrap=True, fontsize=10)
    plt.text(0.1, 0.1, f"Ground Truth:\n{groundtruth_df.iloc[index, 0]}", wrap=True, fontsize=10)
    plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
show_sample(index=0)

AttributeError: 'str' object has no attribute 'unsqueeze'

In [ ]:
# Cell 5: Build model
from models.r2gen import R2GenModel

model = R2GenModel(args,tokenizer)
print("Model initialized successfully!")


/usr/local/lib/python3.8/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.8/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model initialized successfully!


In [ ]:
# Cell 6: Define loss function
from modules.loss import compute_loss

criterion = compute_loss
print("Loss function initialized successfully!")


Loss function initialized successfully!


In [ ]:
# ✅ Grad-CAM Visual Explainer for Medical Report Generation
# This notebook assumes:
# - A model uses VisualExtractor as its CNN backbone
# - Generated and ground truth reports are available
# - Test images are accessed through test_loader

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import cv2
from PIL import Image

# ---------------------------
# Visual Extractor (ResNet based)
# ---------------------------
class VisualExtractor(nn.Module):
    def _init_(self, visual_extractor='resnet101', pretrained=True):
        super(VisualExtractor, self)._init_()
        model = getattr(models, visual_extractor)(pretrained=pretrained)
        self.model = nn.Sequential(*list(model.children())[:-2])
        self.target_layer = self.model[-1]  # last conv layer for Grad-CAM
        self.avg_fnt = torch.nn.AvgPool2d(kernel_size=7, stride=1, padding=0)

    def forward(self, images):
        patch_feats = self.model(images)
        avg_feats = self.avg_fnt(patch_feats).squeeze().reshape(-1, patch_feats.size(1))
        batch_size, feat_size, _, _ = patch_feats.shape
        patch_feats = patch_feats.reshape(batch_size, feat_size, -1).permute(0, 2, 1)
        return patch_feats, avg_feats


# ---------------------------
# Grad-CAM Hook Setup
# ---------------------------
class GradCAM:
    def _init_(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_backward_hook(backward_hook)

    def generate(self, input_tensor, class_idx=None):
        self.model.zero_grad()
        _, avg_feats = self.model(input_tensor)
        if class_idx is None:
            class_idx = avg_feats.argmax().item()
        score = avg_feats[:, class_idx].sum()
        score.backward()
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min())
        cam = cv2.resize(cam, (224, 224))
        return cam


# ---------------------------
# Visualization Utility
# ---------------------------
def visualize_cam_on_image(img_tensor, cam_map):
    img = img_tensor.squeeze().permute(1, 2, 0).cpu().numpy()
    img = (img - img.min()) / (img.max() - img.min())
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_map), cv2.COLORMAP_JET)
    heatmap = np.float32(heatmap) / 255
    overlay = 0.4 * heatmap + 0.6 * img
    plt.imshow(overlay)
    plt.axis('off')
    plt.title("Grad-CAM Overlay")
    plt.show()


# ---------------------------
# Example Usage with test_loader
# ---------------------------
# Assume test_loader provides batches of 2 images per sample (2 views for 1 report)
# and visual_extractor is loaded from the modified class above.

# Example (Put this inside your test loop):
for batch in test_dataloader:

    images, reports = batch[:2]  # Unpack only the first two items
    images = images.view(-1, 3, 224, 224).to(device)  # (B*2, C, H, W)

    extractor = VisualExtractor().to(device)
    grad_cam = GradCAM(extractor, extractor.target_layer)
    cam = grad_cam.generate(images[0].unsqueeze(0))
    visualize_cam_on_image(images[0].cpu(), cam)
    break  # just visualize one sample

AttributeError: 'tuple' object has no attribute 'view'

In [ ]:
# Colab-compatible Grad-CAM notebook using VisualExtractor output (no model retraining)

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import cv2

from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.io import read_image
from torchvision.utils import make_grid

# ---------------------------
# Visual Extractor Definition
# ---------------------------
class VisualExtractor(nn.Module):
    def _init_(self):
        super(VisualExtractor, self)._init_()
        model = models.resnet50(pretrained=True)
        modules = list(model.children())[:-2]  # remove avgpool and fc
        self.model = nn.Sequential(*modules)
        self.avg_fnt = torch.nn.AvgPool2d(kernel_size=7, stride=1, padding=0)

    def forward(self, images):
        patch_feats = self.model(images)  # shape: (B, C, H, W)
        avg_feats = self.avg_fnt(patch_feats).squeeze().reshape(-1, patch_feats.size(1))
        batch_size, feat_size, _, _ = patch_feats.shape
        patch_feats = patch_feats.reshape(batch_size, feat_size, -1).permute(0, 2, 1)
        return patch_feats, avg_feats

# ---------------------------
# Grad-CAM Class Definition
# ---------------------------
class GradCAM:
    def _init_(self, model):
        self.model = model
        self.gradients = None
        self.activations = None
        self.target_layer = self.model.model[-1]  # last conv layer
        self.hook_handles()

    def hook_handles(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_backward_hook(backward_hook)

    def forward(self, x):
        return self.model(x)

    def generate_cam(self, input_tensor):
        self.model.zero_grad()
        output = self.model.model(input_tensor)
        target = output.sum()  # dummy gradient
        target.backward()

        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = torch.nn.functional.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam

# ---------------------------
# Utility for Overlaying CAM
# ---------------------------
def overlay_cam_on_image(img, cam):
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = np.float32(heatmap) / 255
    img = np.float32(img)
    if img.max() > 1:
        img /= 255.0
    overlayed = heatmap + img
    overlayed = overlayed / np.max(overlayed)
    return overlayed

# ---------------------------
# Inference Loop with Grad-CAM
# ---------------------------
def visualize_gradcam(test_dataloader, device):
    extractor = VisualExtractor().to(device)
    grad_cam = GradCAM(extractor)

    for batch in test_dataloader:
        images = batch[0]  # adjust depending on your test_loader structure

        # Handle (B, 2, C, H, W) format
        if images.dim() == 5:
            B = images.size(0)
            images = images.view(B * 2, 3, 224, 224)

        images = images.to(device)
        _ = grad_cam.forward(images)

        for i in range(images.shape[0]):
            cam = grad_cam.generate_cam(images[i].unsqueeze(0))
            img_np = images[i].permute(1, 2, 0).detach().cpu().numpy()
            img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())
            overlay = overlay_cam_on_image(img_np, cam)

            plt.imshow(overlay)
            plt.title(f"Grad-CAM Image {i}")
            plt.axis('off')
            plt.show()

        break  # visualize first batch only

In [ ]:
for batch in test_dataloader:

    images, reports = batch[:2]  # Unpack only the first two items
    images = images.view(-1, 3, 224, 224).to(device)  # (B*2, C, H, W)

    extractor = VisualExtractor().to(device)
    grad_cam = GradCAM(extractor, extractor.target_layer)
    cam = grad_cam.generate(images[0].unsqueeze(0))
    visualize_cam_on_image(images[0].cpu(), cam)
    break  # just visualize one sample

AttributeError: 'tuple' object has no attribute 'view'

In [ ]:
# ---------------------------
# Import Required Libraries
# ---------------------------
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cv2
from torch.utils.data import DataLoader
from torchvision.utils import make_grid

# ---------------------------
# Visual Extractor Definition
# ---------------------------
class VisualExtractor(nn.Module):
    def _init_(self):
        super(VisualExtractor, self)._init_()
        model = models.resnet50(pretrained=True)
        modules = list(model.children())[:-2]
        self.model = nn.Sequential(*modules)
        self.avg_fnt = torch.nn.AvgPool2d(kernel_size=7, stride=1, padding=0)

    def forward(self, images):
        patch_feats = self.model(images)  # (B, C, H, W)
        avg_feats = self.avg_fnt(patch_feats).squeeze().reshape(-1, patch_feats.size(1))
        batch_size, feat_size, _, _ = patch_feats.shape
        patch_feats = patch_feats.reshape(batch_size, feat_size, -1).permute(0, 2, 1)
        return patch_feats, avg_feats

# ---------------------------
# Grad-CAM Class Definition
# ---------------------------
class GradCAM:
    def _init_(self, model):
        self.model = model
        self.model.eval()
        self.gradients = None
        self.activations = None

        target_layer = self.model.model[-1]

        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        target_layer.register_forward_hook(forward_hook)
        target_layer.register_backward_hook(backward_hook)

    def forward(self, x):
        return self.model(x)

    def generate_cam(self, input_tensor):
        self.model.zero_grad()
        patch_feats, _ = self.model(input_tensor)
        target = patch_feats.sum()
        target.backward()

        grads = self.gradients
        acts = self.activations

        weights = grads.mean(dim=(2, 3), keepdim=True)
        cam = (weights * acts).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = torch.nn.functional.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam

# ---------------------------
# Overlay Utility
# ---------------------------
def overlay_cam_on_image(img, cam):
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = np.float32(heatmap) / 255
    img = np.float32(img)
    if img.max() > 1:
        img /= 255.0
    overlayed = heatmap + img
    overlayed = overlayed / np.max(overlayed)
    return overlayed

# ---------------------------
# Visualization Function
# ---------------------------
def run_gradcam_with_reports(test_loader, device, gts, res, max_images=5):
    model = VisualExtractor().to(device)
    gradcam = GradCAM()
    model.eval()

    image_counter = 0

    for batch in test_loader:
        images = batch[0]  # Assume batch is (images, labels) or just images
        if isinstance(images, (list, tuple)):
            images = images[0]  # (B, 2, C, H, W)

        if images.dim() == 5:
            B = images.size(0)
            images = images.view(B * 2, 3, 224, 224)

        images = images.to(device)
        _ = gradcam.forward(images)

        for i in range(images.shape[0]):
            if image_counter >= max_images:
                return

            cam = gradcam.generate_cam(images[i].unsqueeze(0))
            img_np = images[i].permute(1, 2, 0).detach().cpu().numpy()
            img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())
            overlay = overlay_cam_on_image(img_np, cam)

            # Report alignment (assuming order corresponds to test_loader)
            gt_report = gts.iloc[image_counter] if image_counter < len(gts) else "N/A"
            res_report = res.iloc[image_counter] if image_counter < len(res) else "N/A"

            plt.figure(figsize=(6, 6))
            plt.imshow(overlay)
            plt.axis('off')
            plt.title(f"Image {image_counter}")
            plt.show()

            print(f"\033[1mGround Truth:\033[0m {gt_report}")
            print(f"\033[1mGenerated Report:\033[0m {res_report}\n")
            image_counter += 1

In [ ]:
# Load the CSVs
gts = pd.read_csv("/content/gts.csv").squeeze()
res = pd.read_csv("/content/res (1) (1).csv").squeeze()

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Run the Grad-CAM visualization with aligned reports
run_gradcam_with_reports(test_dataloader, device, gts, res, max_images=5)

AttributeError: 'str' object has no attribute 'dim'

In [ ]:
# ================================
# Required Imports
# ================================
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import cv2
import pandas as pd

from torch.utils.data import DataLoader
from torchvision.utils import make_grid

# ================================
# Load Reports from CSVs
# ================================
gts = pd.read_csv("/content/gts.csv", header=None)[0].tolist()
res = pd.read_csv("/content/res (1) (1).csv", header=None)[0].tolist()

# ================================
# Visual Feature Extractor
# ================================
class VisualExtractor(nn.Module):
    def _init_(self):
        super()._init_()
        model = models.resnet50(pretrained=True)
        self.features = nn.Sequential(*list(model.children())[:-2])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        feature_map = self.features(x)  # (B, C, H, W)
        pooled = self.avgpool(feature_map).squeeze(-1).squeeze(-1)  # (B, C)
        return feature_map, pooled

# ================================
# Score-CAM Implementation
# ================================
class ScoreCAM:
    def _init_(self, model):
        self.model = model.eval()
        self.model.to(device)

    def generate_cam(self, input_tensor):
        input_tensor = input_tensor.to(device)
        feature_map, _ = self.model(input_tensor.unsqueeze(0))
        B, C, H, W = feature_map.shape

        cam = torch.zeros((H, W)).to(device)

        for i in range(C):
            fmap = feature_map[0, i, :, :].unsqueeze(0).unsqueeze(0)  # (1,1,H,W)
            fmap_up = torch.nn.functional.interpolate(fmap, size=(224, 224), mode='bilinear', align_corners=False)
            norm_fmap = (fmap_up - fmap_up.min()) / (fmap_up.max() - fmap_up.min() + 1e-8)
            masked_img = input_tensor.unsqueeze(0) * norm_fmap

            _, pred = self.model(masked_img)
            score = pred.squeeze().mean()  # Dummy scalar score

            cam += score * norm_fmap.squeeze()

        cam = torch.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.detach().cpu().numpy()

# ================================
# Overlay Utility
# ================================
def overlay_cam(img_tensor, cam):
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    img = (img - img.min()) / (img.max() - img.min())
    cam = cv2.resize(cam, (img.shape[1], img.shape[0]))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = np.float32(heatmap) / 255
    overlay = heatmap + img
    overlay = overlay / np.max(overlay)
    return overlay

# ================================
# Main Visualization Function
# ================================
def run_scorecam_with_reports(test_loader, gts, res, max_samples=3):
    model = VisualExtractor().to(device)
    scorecam = ScoreCAM()

    print("Running Score-CAM visualization...\n")

    for idx, batch in enumerate(test_dataloader):
        if idx >= max_samples:
            break

        images, _ = batch  # assuming (B, 2, C, H, W)
        if images.dim() == 5:
            B = images.shape[0]
            images = images.view(B * 2, 3, 224, 224)

        for i in range(images.shape[0]):
            img = images[i].to(device)
            cam = scorecam.generate_cam(img)

            overlay = overlay_cam(img, cam)

            # Show the overlay
            plt.figure(figsize=(6, 6))
            plt.imshow(overlay)
            plt.title(f"Grad-CAM Overlay | Sample {idx*2 + i}")
            plt.axis('off')
            plt.show()

            # Print corresponding reports
            report_idx = idx * (images.shape[0]) + i
            if report_idx < len(gts):
                print(f"\033[1m[Generated Report]:\033[0m {res[report_idx]}")
                print(f"\033[1m[Ground Truth]:\033[0m   {gts[report_idx]}")
                print("-" * 100)

# ================================
# Run the full pipeline
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
run_scorecam_with_reports(test_dataloader, gts, res, max_samples=2)

Running Score-CAM visualization...



ValueError: too many values to unpack (expected 2)

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import cv2
import pandas as pd
from tqdm import tqdm

from modules.visual_extractor import VisualExtractor  # ✅ Update this to your actual module

# ===========
# ScoreCAM Class
# ===========
class ScoreCAM:
    def _init_(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.hook = self.target_layer.register_forward_hook(self.save_activations)

    def save_activations(self, module, input, output):
        self.activations = output.detach()

    def generate_cam(self, input_tensor):
        _ = self.model(input_tensor)
        activations = self.activations  # (B, C, H, W)
        b, k, u, v = activations.size()

        cam = torch.zeros((u, v), dtype=torch.float32).to(input_tensor.device)
        for i in range(k):
            # Upsample each activation map to input size
            act = activations[0, i, :, :]
            act = act - act.min()
            act = act / (act.max() + 1e-8)
            upsampled = F.interpolate(act.unsqueeze(0).unsqueeze(0), size=input_tensor.shape[2:], mode='bilinear', align_corners=False)
            score = self.model(input_tensor * upsampled)
            score = score.mean()  # simplification
            cam += score * act

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy()

    def remove_hook(self):
        self.hook.remove()

# ===========
# Helper to overlay CAM
# ===========
def overlay_cam_on_image(img_tensor, cam):
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    img = (img - img.min()) / (img.max() - img.min())  # Normalize
    img = (img * 255).astype(np.uint8)

    cam = cv2.resize(cam, (img.shape[1], img.shape[0]))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)
    return overlay

# ===========
# Main function
# ===========
def run_scorecam_with_reports(test_loader, gts, res, max_samples=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = VisualExtractor().to(device)
    model.eval()

    target_layer = list(model.children())[-1]
    score_cam = ScoreCAM(model, target_layer)

    print("Running Score-CAM visualization...\n")

    with torch.no_grad():
        count = 0
        for batch in tqdm(test_loader):
            if count >= max_samples:
                break

            if isinstance(batch, (tuple, list)):
                images = batch[0].to(device)
            else:
                images = batch.to(device)

            if images.ndim == 5:
                B = images.shape[0]
                images = images.view(B * 2, images.shape[2], images.shape[3], images.shape[4])

            for j, img in enumerate(images):
                if count >= max_samples:
                    break

                cam = score_cam.generate_cam(img.unsqueeze(0))
                overlay = overlay_cam_on_image(img.cpu(), cam)

                idx = count  # each image is separate now
                plt.figure(figsize=(6, 6))
                plt.imshow(overlay)
                plt.axis("off")
                plt.title(f"Generated:\n{res[idx]}\n\nGT:\n{gts[idx]}", fontsize=10)
                plt.tight_layout()
                plt.show()
                count += 1

    score_cam.remove_hook()

# ===========
# Load Reports
# ===========
gts = pd.read_csv("/content/gts.csv", header=None)[0].tolist()
res = pd.read_csv("/content/res (1) (1).csv", header=None)[0].tolist()

# ===========
# Run Score-CAM with Test Dataloader
# ===========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
run_scorecam_with_reports(test_dataloader, gts, res, max_samples=4)

TypeError: VisualExtractor.__init__() missing 1 required positional argument: 'args'

In [ ]:
# Cell 7: Initialize optimizer and learning rate scheduler
from modules.optimizers import build_optimizer, build_lr_scheduler

optimizer = build_optimizer(args, model)
lr_scheduler = build_lr_scheduler(args, optimizer)

print("Optimizer and learning rate scheduler initialized successfully!")


Optimizer and learning rate scheduler initialized successfully!


In [ ]:
trainer_instance = Trainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    args=args,
    lr_scheduler=lr_scheduler,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    test_dataloader=test_dataloader,
)


In [ ]:
trainer_instance.train()

	epoch          : 1
	train_loss     : 3.8025656425035916
	val_loss       : 2.7285509234980534
	test_loss      : 2.350925909506308
Saving checkpoint: results/iu_xray/current_checkpoint.pth ...
Saving current best: model_best.pth ...
	epoch          : 2
	train_loss     : 2.5699060128285334
	val_loss       : 2.216395898869163
	test_loss      : 1.8207258227709178
Saving checkpoint: results/iu_xray/current_checkpoint.pth ...
	epoch          : 3
	train_loss     : 2.173851588139167
	val_loss       : 1.9707117331655402
	test_loss      : 1.560822918608382
Saving checkpoint: results/iu_xray/current_checkpoint.pth ...
	epoch          : 4
	train_loss     : 1.9326188014103816
	val_loss       : 1.8327339762135555
	test_loss      : 1.417222806730786
Saving checkpoint: results/iu_xray/current_checkpoint.pth ...
	epoch          : 5
	train_loss     : 1.7633834096101615
	val_loss       : 1.7263576419729936
	test_loss      : 1.3155957581223667
Saving checkpoint: results/iu_xray/current_checkpoint.pth ...


/workspace/MedGen/modules/trainer.py:110: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  record_table = record_table.append(self.best_recorder['val'], ignore_index=True)
/workspace/MedGen/modules/trainer.py:111: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  record_table = record_table.append(self.best_recorder['test'], ignore_index=True)


In [ ]:
# Initialize tester without metrics
tester = Tester(model, criterion, args, test_dataloader)

# Run testing
tester.test()


02/04/2025 14:55:54 - INFO - modules.tester -   Loading checkpoint: results/iu_xray/current_checkpoint.pth ...
02/04/2025 14:55:57 - INFO - modules.tester -   Start to evaluate in the test set.
37it [00:40,  1.08s/it]


In [ ]:
import pandas as pd

In [ ]:
test_results_df = pd.read_csv(os.path.join(args.save_dir, "res.csv"), header=None)
ground_truths_df = pd.read_csv(os.path.join(args.save_dir, "gts.csv"), header=None)

# Display the results
print("Test Results:")
print(test_results_df.head())
print("Ground Truths:")
print(ground_truths_df.head())

Test Results:
                                                   0
0  the heart is normal in size . the mediastinum ...
1  the heart is normal in size . the mediastinum ...
2  the heart is normal in size . the mediastinum ...
3  the heart is normal in size . the mediastinum ...
4  the heart is normal in size . the mediastinum ...
Ground Truths:
                                                   0
0  normal cardiomediastinal silhouette . there is...
1  lungs are clear . there is no pneumothorax or ...
2  cardiac and mediastinal contours are within no...
3  the heart pulmonary xxxx and mediastinum are w...
4  the cardiomediastinal silhouette is normal in ...


In [ ]:
for images_id, images, reports_ids, reports_masks in test_dataloader:
    print(images_id)  # Print image IDs
    print(reports_ids)  # Print ground truth labels
    break  # Only print one batch


('CXR3030_IM-1405', 'CXR38_IM-1911', 'CXR3957_IM-2022', 'CXR621_IM-2203', 'CXR1347_IM-0225', 'CXR2915_IM-1317', 'CXR34_IM-1644', 'CXR2590_IM-1083', 'CXR1176_IM-0119', 'CXR738_IM-2296', 'CXR2480_IM-1009', 'CXR3222_IM-1522', 'CXR1005_IM-0006', 'CXR3542_IM-1734', 'CXR325_IM-1539', 'CXR2785_IM-1220')
tensor([[  0, 452, 131, 621,   1, 686, 365, 446, 285, 173,   1, 686,  69, 446,
         759, 468,  20, 376, 518, 240,   1, 686, 365, 446, 522,   1, 686, 365,
         446,  30, 111,  25, 605,   1,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [  0, 407,  69, 150,   1, 686, 365, 446, 522, 476, 518, 240,   1, 685,
         315,  50, 423,  69, 753, 452, 389,   1, 111, 652,  69, 356,   1,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0],
        [  0, 129,  50, 422, 1

In [ ]:
output = model(images, mode='sample')
print(output)  # Inspect raw model outputs


RuntimeError: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor

In [ ]:
sample_output = output.cpu().numpy()
decoded_reports = model.tokenizer.decode_batch(sample_output)
print(decoded_reports)  # Check if decoding works as expected


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()